In [20]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np



In [21]:
documents = [
    "Lions are carnivorous mammals. They mainly eat animals such as zebras, wildebeest, antelopes, and buffalo.",
    
    "Lions are found primarily in sub-Saharan Africa, with a small population of Asiatic lions in India.",
    
    "Lions usually live around 10 to 15 years in the wild.",
    
    "Elephants are herbivores and mainly eat grass, leaves, bark, roots, and fruit.",
    
    "Giraffes are herbivores and mainly feed on leaves from trees such as acacia trees.",
    
    "Tigers are carnivores and hunt animals such as deer and wild boar."
]

In [22]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [23]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

generator_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [24]:
question = "What do lions eat?"

In [25]:
question_embedding = embedding_model.encode(
    [question],
    normalize_embeddings=True
)

In [26]:

dimension = question_embedding.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(
    np.array(question_embedding).astype("float32")
)

In [27]:
scores, indices = index.search(
    np.array(question_embedding).astype("float32"),
    k=1
)

In [28]:
retrieved_document = documents[indices[0][0]]

print(retrieved_document)

Lions are carnivorous mammals. They mainly eat animals such as zebras, wildebeest, antelopes, and buffalo.


In [32]:
prompt = f"""
Use the context below to answer the question clearly and specifically.

Context:
{retrieved_document}

Question:
{question}

Give a complete answer in one sentence and include specific examples from the context.

Answer:
"""

In [33]:
inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

In [34]:
outputs = generator_model.generate(
    **inputs,
    max_new_tokens=100
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

animals such as zebras, wildebeest, antelopes, and buffalo
